In [10]:
import pandas as pd
import numpy as np
import xarray as xr
import glob
import os
import re
from scipy.interpolate import griddata

In [ ]:
#组装

k = np.load('../data_pred/pre.npy').T
r = np.load('../data2/eof.npy')


x = np.zeros((23,2324)) 
for i in range(20): #方案1
    a = r[i,:]
    b = k[i]
    e = np.zeros((23,2324))
    for j in range(len(b)):
        bj = float(b[j])
        e[j,:] = a*bj
    x = x + e

prm_coord = xr.open_dataarray('../123456789.nc')
x1 = xr.DataArray(x,coords=prm_coord.coords)
print(x1)
x1['time'] = range(1994,2017)

<xarray.DataArray (time: 23, Stn_No: 2324)> Size: 428kB
array([[-0.12708422, -0.09839957, -0.02673878, ..., -0.0526154 ,
        -0.06248745,  0.08474467],
       [-0.03686501, -0.03401977,  0.00654137, ...,  0.03022194,
        -0.01599966, -0.07497875],
       [-0.07954298, -0.01120451, -0.01029432, ...,  0.09913485,
        -0.00140795,  0.01425493],
       ...,
       [-0.12760102, -0.10738952, -0.10666863, ..., -0.13544258,
        -0.18381655, -0.10218865],
       [-0.03616822,  0.04632703, -0.00844561, ...,  0.0248931 ,
        -0.05726637, -0.01344631],
       [ 0.04679032, -0.00922638, -0.03448542, ...,  0.00579335,
         0.10640037,  0.03610865]], shape=(23, 2324))
Coordinates:
  * Stn_No   (Stn_No) int64 19kB 50136 50246 50247 50349 ... 59954 59981 59985
  * time     (time) datetime64[ns] 184B 1994-01-01 1995-01-01 ... 2016-01-01


In [14]:
#求距平百分率
df = pd.read_csv('../result.csv')
# Step 1: Calculate mean precipitation for each name
mean_precipitation = df.groupby('Stn_No')['Precip'].mean()
# Step 2: Calculate anomaly for each name and year
df['obs_anomaly'] = df.groupby('Stn_No')['Precip'].transform(lambda x: x - x.mean())

# Step 3: Calculate percent deviation for each name and year
df['obs_percent'] = df.apply(lambda row: (row.obs_anomaly / mean_precipitation[row['Stn_No']]), axis=1)

print(df)

       Stn_No  time   Lat   Long  Precip  obs_anomaly  obs_percent
0       50136  1994  5328  12222    1807  -898.652174    -0.332139
1       50136  1995  5328  12222    2642   -63.652174    -0.023526
2       50136  1996  5328  12222    2783    77.347826     0.028587
3       50136  1997  5258  12231    2291  -414.652174    -0.153254
4       50136  1998  5258  12231    2745    39.347826     0.014543
...       ...   ...   ...    ...     ...          ...          ...
55253   59985  2012  1632  11137    3679  -864.695652    -0.190307
55254   59985  2013  1632  11137    7612  3068.304348     0.675288
55255   59985  2014  1632  11137    7193  2649.304348     0.583073
55256   59985  2015  1632  11137    3655  -888.695652    -0.195589
55257   59985  2016  1632  11137    1950 -2593.695652    -0.570834

[55258 rows x 7 columns]


In [15]:

x1_df = x1.to_dataframe(name='prep')

# 重置索引
x1_df.reset_index(inplace=True)

# 合并 dataframe。
# 注意：在此假设在 'Stn_No' 和 'time' 上进行合并是正确的，实际情况可能不同。
df = pd.merge(df, x1_df, on=['Stn_No', 'time'], how='left')

In [ ]:
df = df.rename(columns={'time': 'year'})

In [ ]:
#计算acc

df['X'] = df['obs_percent']
df['Y'] = df['prep']

df['XY'] = df['X'] * df['Y']
sum_XY = df.groupby('year')['XY'].sum()

df['X_square'] = df['X'] ** 2
df['Y_square'] = df['Y'] ** 2
sum_X_square = df.groupby('year')['X_square'].sum()
sum_Y_square = df.groupby('year')['Y_square'].sum()

# multiply and take square root
sqrt_sum = (sum_X_square * sum_Y_square) ** 0.5

result = sum_XY / sqrt_sum
zero_down = len(result[result < 0])
print(result)
print(zero_down)
print(result.mean())
result.to_csv('./观测预测acc2.csv')


year
1994   -0.290928
1995    0.441841
1996    0.151942
1997    0.456820
1998   -0.317537
1999    0.504997
2000    0.187580
2001    0.619667
2002    0.349664
2003    0.401082
2004   -0.103632
2005    0.255107
2006    0.436707
2007    0.297657
2008    0.303891
2009    0.243300
2010    0.443598
2011    0.368098
2012    0.281560
2013   -0.016058
2014    0.331740
2015    0.460908
2016    0.313783
dtype: float64
4
0.2661646014794069
